# 09 — Paper Tables and Figures (07g/08g pipeline only)

Generates every table and figure for a manuscript scoped to the `07g`
(capacity-revision, `pool_anchor` readout) pipeline: Schemes A-G plus
their B+-F+ ablations, and `08g`/`08g_v2`'s post-hoc explanations.

**Read-only.** This notebook trains nothing and loads no model
checkpoints -- it only reads the CSV/JSON/parquet artifacts `07g`,
`08g`, and (optionally) `08g_v2` already wrote to `OUTPUTS_DIR`. It
never modifies anything under `src/` or `configs/`; all output goes to
two new directories under `OUTPUTS_DIR`:

- `paper_tables/` -- one `.csv` + one `.md` per table
- `paper_figures/` -- one `.pdf` + one `.png` per figure

**Run `07g` (required) and `08g` (required for the explainability
tables/figures) first.** `08g_v2` is optional -- its feature-level
tables/figures are skipped gracefully (with a printed note) if that
notebook hasn't been run.

**Style.** House Nature-style conventions (final-size canvas, trimmed
spines, muted palette + one focal series, direct labelling, no
gridlines) via a small `ps` helper defined inline below (Cell 4) --
kept in-notebook rather than as a separate `src/` file, so this
notebook is fully self-contained. Figures with repeated runs (5 repeats
per scheme) use a dotplot (individual repeats + mean +/- s.d.), not a
bar chart, so near-ties stay visible instead of being flattened.
**Assumption** (not asked interactively, since this is unattended
notebook code, not a one-off script): deliver both rendered figures and
the code that made them, one figure per comparison (not one giant
multi-panel composite), with panel letters on the two multi-scenario
small-multiple grids (F4 training curves, F5 confusion matrices) and no
letters on the single-panel figures.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running on Colab -- skipping drive mount (paths.yaml must already resolve locally).")

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn pyyaml pyarrow tabulate

## 1. Paths, config, output directories

Same `paths.yaml` / `eval_capacity_revision.yaml` / `model_capacity_revision.yaml` and the same `CHECKPOINT_DIR`/`METRICS_DIR`/`EXPLAIN_DIR` names `07g`/`08g`/`08g_v2` already write to -- this notebook only reads from them.

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_capacity_revision.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_capacity_revision.yaml") as f:
    model_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
COMBINED_PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])

# Same dir names 07g/08g/08g_v2 already use -- read-only from here on.
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_capacity_revision"
METRICS_DIR = OUTPUTS_DIR / "metrics_capacity_revision"
EXPLAIN_DIR = OUTPUTS_DIR / "explain_capacity_revision"

# New, notebook-09-only output dirs -- nothing else writes here.
TABLE_DIR = OUTPUTS_DIR / "paper_tables"
FIG_DIR = OUTPUTS_DIR / "paper_figures"
for d in (TABLE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

HEAD_DEPTH = model_cfg.get("head_depth", "mlp2")
NORMAL_SCENARIOS = ["A", "B", "C", "D", "E", "F", "G"]
FUSION_SCENARIOS = ["B", "C", "D", "E", "F"]  # ablation variants exist for these only

print("CHECKPOINT_DIR:", CHECKPOINT_DIR, "exists:", CHECKPOINT_DIR.exists())
print("METRICS_DIR:   ", METRICS_DIR, "exists:", METRICS_DIR.exists())
print("EXPLAIN_DIR:   ", EXPLAIN_DIR, "exists:", EXPLAIN_DIR.exists())
print("TABLE_DIR:     ", TABLE_DIR)
print("FIG_DIR:       ", FIG_DIR)

## 2. Small helpers

Every loader prints a clear skip message (which upstream notebook/cell to run) instead of crashing, since this notebook is meant to be re-run incrementally as more of `07g`/`08g`/`08g_v2` finishes.

In [ ]:
import json
import pandas as pd
import numpy as np


def load_csv(path, note=""):
    if not Path(path).exists():
        print(f"  [skip] {path} not found.{(' ' + note) if note else ''}")
        return None
    df = pd.read_csv(path)
    print(f"  [ok]   {path} ({len(df)} rows)")
    return df


def load_json(path, note=""):
    if not Path(path).exists():
        print(f"  [skip] {path} not found.{(' ' + note) if note else ''}")
        return None
    return json.loads(Path(path).read_text())


def save_table(df, name, caption=""):
    """Writes name.csv + name.md (a Markdown table with an optional
    caption line) into TABLE_DIR. Returns df unchanged for chaining."""
    if df is None or len(df) == 0:
        print(f"  [skip] table '{name}' -- no data.")
        return df
    csv_path = TABLE_DIR / f"{name}.csv"
    md_path = TABLE_DIR / f"{name}.md"
    df.to_csv(csv_path, index=False)
    with open(md_path, "w") as f:
        if caption:
            f.write(f"**{caption}**\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n")
    print(f"  [saved] {csv_path.name} + {md_path.name} ({len(df)} rows)")
    return df


def best_repeat_for(tag):
    """Reads {tag}_best_model_meta.json's 'repeat' field -- the ONE
    repeat whose weights survive as {tag}_best_model.pt (selected by
    val_pr_auc, never test -- see train.py). Returns None if missing
    (e.g. scenario G, which has no GNN checkpoint)."""
    meta = load_json(CHECKPOINT_DIR / f"{tag}_best_model_meta.json")
    return meta.get("repeat") if meta else None

## 3. House figure style (inline, self-contained)

Same conventions as the `paper-figure-style` skill's `paperstyle.py` module, copied in directly (not as a separate `src/` file) so this notebook has zero external dependencies beyond the pip installs above.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns


class _PaperStyle:
    FULL_W = 7.0
    HALF_W = 3.4
    COL_W = 4.6

    FS_TICK = 6.5
    FS_LABEL = 7.5
    FS_TITLE = 8.0
    FS_LETTER = 9.5
    FS_LEGEND = 6.0
    FS_ANNOT = 5.8

    MUTED = ["#9fd4c0", "#c3b49a", "#8a7358", "#9aa4cd", "#4a4a73",
             "#8ecae0", "#f2a58c", "#3f8f7d"]
    FOCAL = "#8c2f2f"
    MINIMAL = ["#c9c9c9", "#a8a8a8", "#878787", "#666666", "#3f8f7d"]

    ALPHA_OBS = 0.40
    LW_SPINE = 0.7
    LW_ERR = 1.1
    MS_OBS = 3.2
    MS_MEAN = 5.0
    LW_CELL = 0.5
    CELL_EDGE = "white"

    CMAP_SEQ = "magma"
    CMAP_DIV = "RdBu_r"

    def apply(self, font="Liberation Sans"):
        sns.set_theme(style="ticks")
        mpl.rcParams.update({
            "font.family": "sans-serif",
            "font.sans-serif": [font, "Arial", "Helvetica", "DejaVu Sans"],
            "font.size": self.FS_TICK,
            "axes.labelsize": self.FS_LABEL,
            "axes.titlesize": self.FS_TITLE,
            "xtick.labelsize": self.FS_TICK,
            "ytick.labelsize": self.FS_TICK,
            "legend.fontsize": self.FS_LEGEND,
            "axes.linewidth": self.LW_SPINE,
            "axes.grid": False,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
            "axes.titlelocation": "center",
            "axes.titlepad": 4,
            "axes.labelpad": 3,
            "xtick.direction": "out",
            "ytick.direction": "out",
            "xtick.major.size": 2.4,
            "ytick.major.size": 2.4,
            "xtick.major.width": self.LW_SPINE,
            "ytick.major.width": self.LW_SPINE,
            "xtick.major.pad": 2,
            "ytick.major.pad": 2,
            "lines.linewidth": 1.0,
            "legend.frameon": False,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
            "savefig.pad_inches": 0.02,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        })

    def finish(self, ax, trim=True, offset=None):
        xt, xl = ax.get_xticks(), [t.get_text() for t in ax.get_xticklabels()]
        yt, yl = ax.get_yticks(), [t.get_text() for t in ax.get_yticklabels()]
        xlim, ylim = ax.get_xlim(), ax.get_ylim()
        sns.despine(ax=ax, top=True, right=True, trim=trim, offset=offset)
        if trim:
            if len(ax.get_xticks()) == 0 and len(xt):
                ax.set_xticks(xt)
                if any(xl):
                    ax.set_xticklabels(xl)
            if len(ax.get_yticks()) == 0 and len(yt):
                ax.set_yticks(yt)
                if any(yl):
                    ax.set_yticklabels(yl)
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)

    def panel_letter(self, fig, ax, letter, dx=-0.085, dy=1.06):
        ax.text(dx, dy, letter, transform=ax.transAxes,
                 fontsize=self.FS_LETTER, fontweight="bold", ha="left", va="bottom")

    def sparse_yticks(self, ax, n=5):
        ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=n, prune=None))

    def rotate_xlabels(self, ax, rotation=45, ha="right"):
        for lab in ax.get_xticklabels():
            lab.set_rotation(rotation)
            lab.set_ha(ha)
            lab.set_rotation_mode("anchor")

    def heatmap_axes(self, ax, frame=False):
        for s in ax.spines.values():
            s.set_visible(frame)
            if frame:
                s.set_linewidth(self.LW_SPINE)
        ax.tick_params(length=0, pad=2)

    def annot_color(self, value, vmin, vmax, cmap, threshold=0.55):
        import matplotlib.colors as mcolors
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        r, g, b, _ = mpl.colormaps[cmap](norm(value))
        lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
        return "white" if lum < threshold else "#1a1a1a"

    def slim_colorbar(self, fig, mappable, ax, label="", n_ticks=5):
        from mpl_toolkits.axes_grid1 import make_axes_locatable
        div = make_axes_locatable(ax)
        cax = div.append_axes("right", size="3.0%", pad=0.02)
        cb = fig.colorbar(mappable, cax=cax)
        cb.outline.set_visible(False)
        cb.ax.tick_params(length=1.8, width=self.LW_SPINE, labelsize=self.FS_TICK, pad=1.5)
        cb.set_label(label, fontsize=self.FS_LABEL, labelpad=3)
        cb.locator = mpl.ticker.MaxNLocator(nbins=n_ticks)
        cb.update_ticks()
        return cb

    def save(self, fig, name):
        fig.savefig(FIG_DIR / f"{name}.pdf")
        fig.savefig(FIG_DIR / f"{name}.png")
        print(f"  [saved] {name}.pdf + {name}.png")


ps = _PaperStyle()
ps.apply()
SCHEME_COLOR = {s: ps.MUTED[i % len(ps.MUTED)] for i, s in enumerate(NORMAL_SCENARIOS)}
rng = np.random.default_rng(42)

## 4. Load core data sources

Everything below reads a file `07g`/`08g` already wrote. Nothing here re-runs training or re-derives features.

In [ ]:
print("Dataset index (for T1):")
index_df = None
index_path = COMBINED_PROCESSED_DIR / "dataset_index.parquet"
if index_path.exists():
    index_df = pd.read_parquet(index_path)
    print(f"  [ok]   {index_path} ({len(index_df)} points, {index_df['city'].nunique()} cities)")
else:
    print(f"  [skip] {index_path} not found -- run 01-05 first.")

print("\n07g main results (T4a, F2):")
summary_df = load_csv(METRICS_DIR / "all_scenarios_summary_capacity_revision.csv",
                       "Run 07g's aggregate cell.")

print("\n07g best-repeat val/test metrics (T4b):")
best_repeat_df = load_csv(METRICS_DIR / "val_test_metrics_best_repeat.csv",
                           "Run 07g's best-repeat aggregation cell.")

print("\n07g per-point raw test predictions (F3, F5, F9, per-repeat metrics for F2):")
raw_pred_df = load_csv(METRICS_DIR / "raw_test_predictions_all_scenarios.csv",
                        "Run 07g's raw-predictions aggregation cell.")

print("\n08g node/edge-type explanations (T6, F6, F7):")
explain_normal_df = load_csv(EXPLAIN_DIR / "explanations_capacity_revision_normal.csv",
                              "Run 08g.")
explain_ablation_df = load_csv(EXPLAIN_DIR / "explanations_capacity_revision_ablation.csv",
                                "Run 08g.")
type_pivot_normal_df = load_csv(EXPLAIN_DIR / "type_importance_summary_normal.csv", "Run 08g.")
type_topn_normal_df = load_csv(EXPLAIN_DIR / "type_importance_top5_normal.csv", "Run 08g.")
type_pivot_ablation_df = load_csv(EXPLAIN_DIR / "type_importance_summary_ablation.csv", "Run 08g.")

print("\n08g_v2 feature-level explanations (T7, F8 -- optional):")
feature_pivot_normal_df = load_csv(EXPLAIN_DIR / "feature_importance_summary_normal.csv",
                                    "Optional -- run 08g_v2 to enable.")
feature_topn_normal_df = load_csv(EXPLAIN_DIR / "feature_importance_top5_normal.csv",
                                   "Optional -- run 08g_v2 to enable.")

print("\nRaw positive/negative point CSVs per city (F1, T1 road-type match):")
pos_dfs, neg_dfs = [], []
for city in CITIES:
    pc = paths_cfg["per_city"][city]
    p = load_csv(pc["positive_points_csv"])
    n = load_csv(pc["negative_points_csv"])
    if p is not None:
        p = p.copy(); p["city"] = city; pos_dfs.append(p)
    if n is not None:
        n = n.copy(); n["city"] = city; neg_dfs.append(n)
positive_points_df = pd.concat(pos_dfs, ignore_index=True) if pos_dfs else None
negative_points_df = pd.concat(neg_dfs, ignore_index=True) if neg_dfs else None

---
# Tables

## T1 -- Dataset summary per city

In [ ]:
t1_rows = []
if index_df is not None:
    for city, g in index_df.groupby("city"):
        row = {"city": city, "n_total": len(g), "n_positive": int(g["label"].sum()),
               "n_negative": int((g["label"] == 0).sum())}
        if positive_points_df is not None and "highway" in positive_points_df.columns:
            city_pos = positive_points_df[positive_points_df["city"] == city]
            if len(city_pos):
                row["top_road_type"] = city_pos["highway"].value_counts().idxmax()
                row["top_road_type_share"] = round(
                    city_pos["highway"].value_counts(normalize=True).max(), 3)
        t1_rows.append(row)

t1_df = pd.DataFrame(t1_rows)
save_table(t1_df, "T1_dataset_summary_per_city",
           "Table 1. Dataset composition per city (pooled 07g dataset, post out-of-range filtering).")
display(t1_df)

## T2 -- Node feature schema (raw dims, from config)

Structural dims (position=2, area=1, ...) are architectural constants; vocabulary sizes and embedding widths are read live from `model_cfg` and the post-`04b` vocab caches, so this table can never silently drift from what `07g` actually trained.

In [ ]:
cat_embed_dim = model_cfg.get("cat_embed_dim", 4)
building_type_embed_dim = model_cfg.get("building_type_embed_dim", 16)
highway_embed_dim = model_cfg.get("highway_embed_dim", 8)

building_type_vocab_size = None
highway_vocab_size = None
_ref_cache_dir = INTERIM_DIR / "osm_cache" / CITIES[0]
_bt_vocab = load_json(_ref_cache_dir / "building_type_vocab.json")
_hw_vocab = load_json(_ref_cache_dir / "highway_vocab.json")
if _bt_vocab is not None:
    building_type_vocab_size = len(_bt_vocab)
if _hw_vocab is not None:
    highway_vocab_size = len(_hw_vocab)

t2_rows = [
    {"graph": "egocentric", "node_type": "ego (viewpoint)", "raw_dim": 5,
     "composition": "svf, enclosure, entropy, pos_x, pos_y"},
    {"graph": "egocentric", "node_type": "signage / light_pole / road_marking", "raw_dim": 3 + cat_embed_dim,
     "composition": f"pos_x, pos_y, area, class_embed(d={cat_embed_dim})"},
    {"graph": "egocentric", "node_type": "building / vegetation", "raw_dim": 3,
     "composition": "pos_x, pos_y, area"},
    {"graph": "allocentric", "node_type": "focal (incident)", "raw_dim": 4 + highway_embed_dim,
     "composition": f"fraction_along, isovist_area, isovist_compactness, isovist_occlusivity, "
                     f"highway_embed(d={highway_embed_dim}, vocab={highway_vocab_size})"},
    {"graph": "allocentric", "node_type": "building", "raw_dim": 6 + building_type_embed_dim + 8,
     "composition": f"area, perimeter, compactness, elongation, orientation, shape_index, "
                     f"building_type_embed(d={building_type_embed_dim}, vocab={building_type_vocab_size}), "
                     f"height_proj(d=4), levels_proj(d=4)"},
    {"graph": "allocentric", "node_type": "intersection", "raw_dim": 2,
     "composition": "betweenness, orientation_entropy"},
    {"graph": "allocentric", "node_type": "peer_incident (ablation-only)", "raw_dim": 1,
     "composition": "constant placeholder"},
]
t2_df = pd.DataFrame(t2_rows)
save_table(t2_df, "T2_node_feature_schema",
           "Table 2. Node types and raw feature dimensionality (07g/pool_anchor pipeline).")
display(t2_df)

## T3 -- Architecture and training hyperparameters (from config)

In [ ]:
t3_source = {
    "hidden_dim (d_h)": model_cfg.get("hidden_dim"),
    "attention heads (H)": model_cfg.get("heads"),
    "message-passing layers (L)": model_cfg.get("svg_layers"),
    "encoder dropout": model_cfg.get("dropout"),
    "head_depth": HEAD_DEPTH,
    "head_hidden": eval_cfg.get("head_hidden") or model_cfg.get("head_hidden"),
    "head_dropout": eval_cfg.get("head_dropout", 0.5),
    "fusion_dim": model_cfg.get("fusion_dim"),
    "cat_embed_dim": cat_embed_dim,
    "building_type_embed_dim": building_type_embed_dim,
    "highway_embed_dim": highway_embed_dim,
    "optimizer": "AdamW",
    "learning rate": 1e-3,
    "weight decay": 1e-3,
    "val_frac": eval_cfg.get("val_frac"),
    "test_frac": 0.10,
    "epoch_cap": 100,
    "n_repeats": 5,
    "decision threshold": 0.5,
}
t3_df = pd.DataFrame([{"parameter": k, "value": v} for k, v in t3_source.items()])
save_table(t3_df, "T3_hyperparameters", "Table 3. Architecture and training hyperparameters (07g).")
display(t3_df)

## T4 -- Main results

Two variants: **T4a** is the descriptive mean +/- s.d. across all 5 repeats per scheme (what `07g`'s own aggregate cell produces); **T4b** is the val/test metric pair for the ONE repeat actually kept as `{tag}_best_model.pt` -- the checkpoint `08g`'s explanations are computed from. Report both: T4a for "how stable is this scheme across re-splits," T4b for "what does the specific reported/explained model score."

In [ ]:
t4a_df = None
if summary_df is not None:
    keep_cols = ["scenario"] + [c for c in summary_df.columns
                                 if c.split("_mean")[0].split("_std")[0]
                                 in ("accuracy", "precision", "recall", "f1", "pr_auc", "auroc")]
    t4a_df = summary_df[keep_cols].copy()
    t4a_df = save_table(t4a_df, "T4a_main_results_5repeat_mean_std",
                         "Table 4a. Mean +/- s.d. across 5 repeats, per scheme (07g).")
    display(t4a_df)

In [ ]:
t4b_df = None
if best_repeat_df is not None:
    metric_cols = [c for c in ("accuracy", "precision", "recall", "f1", "pr_auc", "auroc")
                   if c in best_repeat_df.columns]
    t4b_df = best_repeat_df[["scenario", "split", "repeat"] + metric_cols].copy()
    t4b_df = save_table(t4b_df, "T4b_best_repeat_val_test",
                         "Table 4b. Val/test metrics for the single best-repeat checkpoint per scheme (07g/08g).")
    display(t4b_df)

## T5 -- Ablation comparison (B-F vs B+-F+)

In [ ]:
t5_df = None
if t4a_df is not None:
    base = t4a_df[t4a_df["scenario"].isin(FUSION_SCENARIOS)].set_index("scenario")
    abl = t4a_df[t4a_df["scenario"].str.endswith("_ablation")].copy()
    abl["scenario"] = abl["scenario"].str.replace(f"_{HEAD_DEPTH}", "", regex=False) \
                                      .str.replace("_ablation", "", regex=False)
    abl = abl.set_index("scenario")
    common = [s for s in FUSION_SCENARIOS if s in base.index and s in abl.index]
    rows = []
    for s in common:
        for metric in ("pr_auc_mean", "auroc_mean", "accuracy_mean"):
            if metric in base.columns and metric in abl.columns:
                rows.append({"scenario": s, "metric": metric.replace("_mean", ""),
                             "base": base.loc[s, metric], "ablation": abl.loc[s, metric],
                             "delta": abl.loc[s, metric] - base.loc[s, metric]})
    t5_df = pd.DataFrame(rows)
    t5_df = save_table(t5_df, "T5_ablation_comparison",
                        "Table 5. Non-ablation vs. ablation (crash_history/peer_incident) mean scores, per scheme.")
    display(t5_df)

## T6 -- Scheme-level node/edge-type importance (top-5, GNNExplainer)

In [ ]:
t6_df = None
if type_topn_normal_df is not None:
    t6_df = type_topn_normal_df[type_topn_normal_df["source"] == "gnnexplainer"].copy()
    t6_df = save_table(t6_df, "T6_type_importance_top5",
                        "Table 6. Top-5 most important node/edge types per scheme (GNNExplainer, 08g).")
    display(t6_df)

## T7 -- Feature-level importance (top-5, optional -- requires 08g_v2)

In [ ]:
t7_df = None
if feature_topn_normal_df is not None:
    t7_df = save_table(feature_topn_normal_df, "T7_feature_importance_top5",
                        "Table 7. Top-5 most important named feature components per scheme (GNNExplainer, 08g_v2).")
    display(t7_df)
else:
    print("08g_v2 not run -- T7 skipped.")

---
# Figures

## F1 -- Road-type distribution: positive vs. generated negatives

Grouped bars (§2.2's matching claim, pooled across all cities).

In [ ]:
if positive_points_df is not None and negative_points_df is not None \
        and "highway" in positive_points_df.columns and "highway" in negative_points_df.columns:
    pos_prop = positive_points_df["highway"].value_counts(normalize=True)
    neg_prop = negative_points_df["highway"].value_counts(normalize=True)
    road_types = sorted(set(pos_prop.index) | set(neg_prop.index),
                         key=lambda t: -pos_prop.get(t, 0))[:10]

    x = np.arange(len(road_types))
    w = 0.38
    fig, ax = plt.subplots(figsize=(ps.COL_W, 2.6))
    ax.bar(x - w / 2, [pos_prop.get(t, 0) for t in road_types], w,
           color=ps.MUTED[4], label="positive")
    ax.bar(x + w / 2, [neg_prop.get(t, 0) for t in road_types], w,
           color=ps.FOCAL, label="negative (generated)")
    ax.set_xticks(x)
    ax.set_xticklabels(road_types)
    ps.rotate_xlabels(ax)
    ax.set_ylabel("proportion of points")
    ax.legend(loc="upper right")
    ps.sparse_yticks(ax)
    ps.finish(ax)
    fig.tight_layout()
    ps.save(fig, "F1_road_type_distribution")
    plt.show()
else:
    print("positive/negative point CSVs (with a 'highway' column) not found -- F1 skipped.")

## F2 -- Main performance comparison (dotplot, PR-AUC and AUROC)

Per-repeat metrics are recomputed directly from `raw_test_predictions_all_scenarios.csv` (point-level prob/label), not read back from the pre-aggregated summary CSV, so every repeat's individual score is available for the dotplot's jittered-observation layer -- not just its mean/std.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score

per_repeat_metrics_df = None
if raw_pred_df is not None:
    rows = []
    for (scenario, repeat), g in raw_pred_df.groupby(["scenario", "repeat"]):
        if g["label"].nunique() < 2:
            continue
        rows.append({
            "scenario": scenario, "repeat": repeat,
            "pr_auc": average_precision_score(g["label"], g["prob"]),
            "auroc": roc_auc_score(g["label"], g["prob"]),
            "accuracy": accuracy_score(g["label"], g["pred"]),
        })
    per_repeat_metrics_df = pd.DataFrame(rows)
    save_table(per_repeat_metrics_df, "per_repeat_metrics_recomputed",
               "Per-repeat metrics recomputed from raw point-level test predictions (feeds F2).")
    display(per_repeat_metrics_df.groupby("scenario")[["pr_auc", "auroc"]].agg(["mean", "std"]))


if per_repeat_metrics_df is not None:
    # tags in raw_pred_df are the full {scenario}_{head_depth} / "G" tag;
    # normalize to the bare scheme letter for plotting.
    per_repeat_metrics_df["scheme"] = per_repeat_metrics_df["scenario"].apply(
        lambda t: "G" if t == "G" else t.split(f"_{HEAD_DEPTH}")[0])
    non_ablation = per_repeat_metrics_df[~per_repeat_metrics_df["scenario"].str.contains("ablation")]
    scenarios_present = [s for s in NORMAL_SCENARIOS if (non_ablation["scheme"] == s).any()]

    fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.8))
    for ax, metric, title in zip(axes, ("pr_auc", "auroc"), ("PR-AUC", "AUROC")):
        for i, s in enumerate(scenarios_present):
            vals = non_ablation.loc[non_ablation["scheme"] == s, metric].values
            color = SCHEME_COLOR.get(s, ps.MUTED[0])
            j = rng.uniform(-0.17, 0.17, len(vals))
            ax.plot(i + j, vals, "o", color=color, alpha=ps.ALPHA_OBS, ms=ps.MS_OBS, mec="none", zorder=2)
            m, sd = vals.mean(), (vals.std(ddof=1) if len(vals) > 1 else 0.0)
            ax.plot([i, i], [m - sd, m + sd], "-", color=color, lw=ps.LW_ERR, zorder=3)
            ax.plot(i, m, "o", color=color, ms=ps.MS_MEAN, mec="none", zorder=4)
        ax.set_xticks(range(len(scenarios_present)))
        ax.set_xticklabels(scenarios_present)
        ax.set_ylabel(title)
        ax.set_title(title)
        ps.sparse_yticks(ax)
        ps.finish(ax)
    fig.tight_layout()
    ps.save(fig, "F2_main_performance_comparison")
    plt.show()
else:
    print("raw_test_predictions_all_scenarios.csv not found -- F2 skipped.")

## F3 -- Precision-recall and ROC curves (best-repeat test split, per scheme)

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

if raw_pred_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.8))
    for i, s in enumerate([sc for sc in NORMAL_SCENARIOS if sc != "G"]):
        tag = f"{s}_{HEAD_DEPTH}"
        best_repeat = best_repeat_for(tag)
        if best_repeat is None:
            continue
        g = raw_pred_df[(raw_pred_df["scenario"] == tag) & (raw_pred_df["repeat"] == best_repeat)]
        if len(g) == 0 or g["label"].nunique() < 2:
            continue
        color = SCHEME_COLOR[s]
        prec, rec, _ = precision_recall_curve(g["label"], g["prob"])
        fpr, tpr, _ = roc_curve(g["label"], g["prob"])
        axes[0].plot(rec, prec, color=color, lw=1.0)
        axes[0].text(rec[-1], prec[-1], f" {s}", color=color, fontsize=ps.FS_LABEL, va="center")
        axes[1].plot(fpr, tpr, color=color, lw=1.0)
        axes[1].text(fpr[-2] if len(fpr) > 1 else 1.0, tpr[-2] if len(tpr) > 1 else 1.0,
                     f" {s}", color=color, fontsize=ps.FS_LABEL, va="center")

    axes[1].plot([0, 1], [0, 1], "--", color="0.7", lw=0.7, zorder=1)
    axes[0].set_xlabel("recall"); axes[0].set_ylabel("precision"); axes[0].set_title("Precision-recall")
    axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].set_title("ROC")
    for ax in axes:
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ps.sparse_yticks(ax); ps.finish(ax)
    fig.tight_layout()
    ps.save(fig, "F3_pr_roc_curves")
    plt.show()
else:
    print("raw_test_predictions_all_scenarios.csv not found -- F3 skipped.")

## F4 -- Training curves (best repeat per scheme, small multiples)

In [ ]:
letters = "abcdefg"
plot_scenarios = [s for s in NORMAL_SCENARIOS if s != "G"]
histories = {}
for s in plot_scenarios:
    tag = f"{s}_{HEAD_DEPTH}"
    best_repeat = best_repeat_for(tag)
    if best_repeat is None:
        continue
    hist = load_json(CHECKPOINT_DIR / f"{tag}_history" / f"repeat{best_repeat}.json")
    if hist:
        histories[s] = hist

if histories:
    ncols = 3
    nrows = -(-len(histories) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ps.FULL_W, 2.3 * nrows), squeeze=False)
    for i, (s, hist) in enumerate(histories.items()):
        ax = axes[i // ncols][i % ncols]
        epochs = [h["epoch"] for h in hist]
        val_acc = [h.get("val_accuracy") for h in hist]
        val_prauc = [h.get("val_pr_auc") for h in hist]
        ax.plot(epochs, val_acc, color=ps.FOCAL, lw=1.0, label="val_accuracy")
        ax.plot(epochs, val_prauc, color=ps.MUTED[3], lw=0.9, label="val_pr_auc")
        ax.set_title(s, fontsize=ps.FS_TITLE, fontweight="bold")
        ax.set_xlabel("epoch"); ax.set_ylabel("metric")
        ps.sparse_yticks(ax); ps.finish(ax)
        ps.panel_letter(fig, ax, letters[i])
        if i == 0:
            ax.legend(loc="lower right", fontsize=ps.FS_LEGEND)
    for j in range(len(histories), nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    fig.tight_layout()
    ps.save(fig, "F4_training_curves")
    plt.show()
else:
    print("No per-repeat history JSONs found -- F4 skipped.")

## F5 -- Confusion matrices (best repeat, test split, small multiples)

In [ ]:
from sklearn.metrics import confusion_matrix

if raw_pred_df is not None:
    cms = {}
    for s in [sc for sc in NORMAL_SCENARIOS if sc != "G"]:
        tag = f"{s}_{HEAD_DEPTH}"
        best_repeat = best_repeat_for(tag)
        if best_repeat is None:
            continue
        g = raw_pred_df[(raw_pred_df["scenario"] == tag) & (raw_pred_df["repeat"] == best_repeat)]
        if len(g) == 0:
            continue
        cms[s] = confusion_matrix(g["label"], g["pred"])

    if cms:
        ncols = 3
        nrows = -(-len(cms) // ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ps.FULL_W, 2.2 * nrows), squeeze=False)
        for i, (s, cm) in enumerate(cms.items()):
            ax = axes[i // ncols][i % ncols]
            im = ax.imshow(cm, cmap=ps.CMAP_SEQ, aspect="auto", interpolation="nearest")
            for (r, c), v in np.ndenumerate(cm):
                ax.text(c, r, str(v), ha="center", va="center", fontsize=ps.FS_ANNOT,
                        color=ps.annot_color(v, cm.min(), cm.max(), ps.CMAP_SEQ))
            ax.set_xticks([0, 1]); ax.set_xticklabels(["pred 0", "pred 1"])
            ax.set_yticks([0, 1]); ax.set_yticklabels(["true 0", "true 1"])
            ax.set_title(s, fontsize=ps.FS_TITLE, fontweight="bold")
            ps.heatmap_axes(ax)
            ps.panel_letter(fig, ax, "abcdefg"[i])
        for j in range(len(cms), nrows * ncols):
            axes[j // ncols][j % ncols].axis("off")
        fig.tight_layout()
        ps.save(fig, "F5_confusion_matrices")
        plt.show()
    else:
        print("No best-repeat predictions found -- F5 skipped.")
else:
    print("raw_test_predictions_all_scenarios.csv not found -- F5 skipped.")

## F6 -- Attention vs. GNNExplainer agreement (per node/edge type)

In [ ]:
if type_pivot_normal_df is not None and {"gnnexplainer", "attention"}.issubset(type_pivot_normal_df.columns):
    sub = type_pivot_normal_df.dropna(subset=["gnnexplainer", "attention"])
    if len(sub):
        kinds = sub["kind"].unique()
        kind_color = {k: ps.MUTED[i] for i, k in enumerate(kinds)}
        fig, ax = plt.subplots(figsize=(ps.HALF_W, ps.HALF_W))
        for k in kinds:
            g = sub[sub["kind"] == k]
            ax.scatter(g["attention"], g["gnnexplainer"], s=10, color=kind_color[k],
                       alpha=0.75, label=k, edgecolor="none")
        lims = [0, max(sub["attention"].max(), sub["gnnexplainer"].max()) * 1.05]
        ax.plot(lims, lims, "--", color="0.7", lw=0.7, zorder=1)
        corr = sub["attention"].corr(sub["gnnexplainer"])
        ax.set_xlabel("mean attention weight")
        ax.set_ylabel("mean GNNExplainer importance")
        ax.set_title(f"r = {corr:.2f}", fontsize=ps.FS_TITLE)
        ax.legend(loc="upper left", fontsize=ps.FS_LEGEND)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ps.sparse_yticks(ax); ps.finish(ax)
        fig.tight_layout()
        ps.save(fig, "F6_attention_vs_gnnexplainer")
        plt.show()
    else:
        print("type_importance_summary_normal.csv has no rows with both signals -- F6 skipped.")
else:
    print("type_importance_summary_normal.csv not found -- F6 skipped.")

## F7 -- Node/edge-type importance heatmap (schemes x types, GNNExplainer)

In [ ]:
if type_pivot_normal_df is not None and "gnnexplainer" in type_pivot_normal_df.columns:
    heat = type_pivot_normal_df.pivot_table(index="type", columns="scenario",
                                             values="gnnexplainer", aggfunc="mean")
    if heat.size:
        fig, ax = plt.subplots(figsize=(ps.COL_W, 0.32 * len(heat) + 1.0))
        im = ax.imshow(heat.values, cmap=ps.CMAP_SEQ, aspect="auto", interpolation="nearest")
        ax.set_xticks(range(heat.shape[1])); ax.set_xticklabels(heat.columns)
        ax.set_yticks(range(heat.shape[0])); ax.set_yticklabels(heat.index)
        if heat.size <= 150:
            vmin, vmax = np.nanmin(heat.values), np.nanmax(heat.values)
            for r in range(heat.shape[0]):
                for c in range(heat.shape[1]):
                    v = heat.values[r, c]
                    if not np.isnan(v):
                        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=ps.FS_ANNOT,
                                color=ps.annot_color(v, vmin, vmax, ps.CMAP_SEQ))
        ax.set_xticks(np.arange(-0.5, heat.shape[1], 1), minor=True)
        ax.set_yticks(np.arange(-0.5, heat.shape[0], 1), minor=True)
        ax.grid(which="minor", color=ps.CELL_EDGE, linewidth=ps.LW_CELL)
        ax.tick_params(which="minor", length=0)
        ps.heatmap_axes(ax)
        ps.slim_colorbar(fig, im, ax, label="mean GNNExplainer importance")
        fig.tight_layout()
        ps.save(fig, "F7_type_importance_heatmap")
        plt.show()
    else:
        print("type_importance_summary_normal.csv pivoted to an empty table -- F7 skipped.")
else:
    print("type_importance_summary_normal.csv not found -- F7 skipped.")

## F8 -- Feature-level importance heatmap (optional -- requires 08g_v2)

In [ ]:
if feature_pivot_normal_df is not None:
    fp = feature_pivot_normal_df.copy()
    fp["feature_label"] = fp["node_type"].astype(str) + "." + fp["feature"].astype(str)
    heat = fp.pivot_table(index="feature_label", columns="scenario", values="mean_importance", aggfunc="mean")
    if heat.size:
        fig, ax = plt.subplots(figsize=(ps.COL_W, 0.28 * len(heat) + 1.0))
        im = ax.imshow(heat.values, cmap=ps.CMAP_SEQ, aspect="auto", interpolation="nearest")
        ax.set_xticks(range(heat.shape[1])); ax.set_xticklabels(heat.columns)
        ax.set_yticks(range(heat.shape[0])); ax.set_yticklabels(heat.index, fontsize=ps.FS_ANNOT)
        ax.set_xticks(np.arange(-0.5, heat.shape[1], 1), minor=True)
        ax.set_yticks(np.arange(-0.5, heat.shape[0], 1), minor=True)
        ax.grid(which="minor", color=ps.CELL_EDGE, linewidth=ps.LW_CELL)
        ax.tick_params(which="minor", length=0)
        ps.heatmap_axes(ax)
        ps.slim_colorbar(fig, im, ax, label="mean importance")
        fig.tight_layout()
        ps.save(fig, "F8_feature_importance_heatmap")
        plt.show()
    else:
        print("feature_importance_summary_normal.csv pivoted to an empty table -- F8 skipped.")
else:
    print("08g_v2 not run -- F8 skipped.")

## F9 -- Calibration (reliability) plot, best repeat, per scheme

In [ ]:
from sklearn.calibration import calibration_curve

if raw_pred_df is not None:
    fig, ax = plt.subplots(figsize=(ps.HALF_W, ps.HALF_W))
    ax.plot([0, 1], [0, 1], "--", color="0.7", lw=0.7, zorder=1)
    for s in [sc for sc in NORMAL_SCENARIOS if sc != "G"]:
        tag = f"{s}_{HEAD_DEPTH}"
        best_repeat = best_repeat_for(tag)
        if best_repeat is None:
            continue
        g = raw_pred_df[(raw_pred_df["scenario"] == tag) & (raw_pred_df["repeat"] == best_repeat)]
        if len(g) < 10 or g["label"].nunique() < 2:
            continue
        frac_pos, mean_pred = calibration_curve(g["label"], g["prob"], n_bins=8, strategy="quantile")
        color = SCHEME_COLOR[s]
        ax.plot(mean_pred, frac_pos, "-o", color=color, ms=2.5, lw=0.9)
        ax.text(mean_pred[-1], frac_pos[-1], f" {s}", color=color, fontsize=ps.FS_LABEL, va="center")
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("observed fraction positive")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ps.sparse_yticks(ax); ps.finish(ax)
    fig.tight_layout()
    ps.save(fig, "F9_calibration")
    plt.show()
else:
    print("raw_test_predictions_all_scenarios.csv not found -- F9 skipped.")

---
## Summary

All tables land in `paper_tables/` as matched `.csv` + `.md` pairs;
all figures land in `paper_figures/` as matched `.pdf` + `.png` pairs.
Nothing under `src/` or `configs/` was read for anything other than
reference values (dimensions, hyperparameters), and nothing there was
written to.

In [ ]:
print("Tables in", TABLE_DIR, ":")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p.name)
print("\nFigures in", FIG_DIR, ":")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p.name)